In [57]:
import glob
import os
from pathlib import Path

import dask.array as da
import napari
import numpy as np
import tifffile as tff
import tqdm
import yaml
from multiview_stitcher import fusion
from multiview_stitcher import spatial_image_utils as si_utils

#from napari_autoscape import convert_single_tile_to_ome_zarr, convert_tiles_to_stitched_ome_zarr
from ome_zarr import (
    OMEZarrHCSPlate,
    OMEZarrImage,
    OMEZarrMultiscale,
    OMEZarrScene,
)
from ome_zarr_models.v06.coordinate_transforms import (
    Affine,
    Axis,
    CoordinateSystem,
    CoordinateSystemIdentifier,
    Translation,
)
from ultralytics import YOLO

from napari_autoscape.focus_detector import (
    FocusLightningModel,
    run_focus_detection,
)
from napari_autoscape.object_detector import object_detection_on_scene

In [3]:
viewer = napari.Viewer()

In [4]:
places = [
    r'D:\UserData\Johannes\20260911\Overview_1',
]

## Read tile images

In [6]:
imgs = []
transforms = []
for place in places:
    tif_files = glob.glob(os.path.join(place, "*.ome.tif"), recursive=True)

    sims = []
    translations = []
    metadata_file = tif_files[0].replace(".ome.tif", "_metadata.txt")
    with open(metadata_file) as f:
        metadata = yaml.safe_load(f)["Summary"]

    for tf in tqdm.tqdm(tif_files):

        tile_index = Path(tf).name.split("MMStack_")[1].replace(".ome.tif", "")

        # index positionlist by label
        positionlist = metadata["InitialPositionList"]
        positionlist = {position["Label"]: position for position in positionlist}
        position = positionlist[tile_index]

        scale = {
            "z": metadata["z-step_um"] if metadata["z-step_um"] != 0 else 1.0,
            "y": metadata["PixelSize_um"],
            "x": metadata["PixelSize_um"]
        }

        array = da.from_array(tff.imread(tf, is_ome=False, is_mmstack=False))
        size_y = array.shape[1] * scale["y"]
        size_x = array.shape[2] * scale["x"]
        size_z = array.shape[0] * scale["z"]

        translation = {
            "z": float(position["DeviceCoordinatesUm"]["ZStage:Z:32"][0]) - size_z // 2,
            "y": - float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][1]) - size_y // 2,
            "x": float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][0]) - size_x // 2,
        }
        translations.append(translation)

        sims.append(
            si_utils.get_sim_from_array(
                array,
                dims=["z", "y", "x"],
                scale=scale,
                translation=translation,
                transform_key="stage_metadata",
                )
        )

    image = fusion.fuse(sims, transform_key="stage_metadata", fusion_func=fusion.max_fusion, backend="cupy")

    ngff_img = OMEZarrImage(image.data.squeeze(), axes="zyx", scale=scale,
                            axes_units={"z": "micrometer", "y": "micrometer", "x": "micrometer"}, name=Path(place).stem)
    ms_img = OMEZarrMultiscale(ngff_img, channel_names=["Brightfield"], channel_colors=["808080"])
    imgs.append(ms_img)

    z_min = min([t["z"] for t in translations])
    y_min = min([t["y"] for t in translations])
    x_min = min([t["x"] for t in translations])

    transforms.append({
        "type": "translation",
        "translation": (z_min, y_min, x_min),
        "input": {"name": "physical", "path": ms_img.name},
        "output": {"name": "world"}
    })


100%|██████████| 342/342 [00:32<00:00, 10.58it/s]


In [7]:
world_cs = CoordinateSystem(name="world", axes=[
    Axis(name="z", type="space", unit="micrometer"),
    Axis(name="y", type="space", unit="micrometer"),
    Axis(name="x", type="space", unit="micrometer")])

scene = OMEZarrScene(
    images=imgs,
    coordinate_systems=(world_cs,),
    coordinate_transformations=transforms
)

scene_path = r"Z:\Collaborations\aif-botton-collaboration\Johannes\20260911\scene.ome.zarr"
scene.to_ome_zarr(scene_path, compute=True, overwrite=True)

c:\Users\aifadmin\Documents\Github\autoSCAPE\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `tuple[Dataset, ...]` - serialized value may not be as expected [field_name='datasets', input_value=[Dataset(path='s0', coord....438733060921248)))),))], input_type=list])
  return self.__pydantic_serializer__.to_python(


[]

## View

In [8]:
viewer.layers.clear()
viewer.open(scene_path, plugin="napari-ome-zarr")

Root group {'ome': {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': 'Overview_1'}, 'output': {'name': 'world'}, 'translation': [25.02000000000001, 2427.2011200000015, 2968.8107800000002]}], 'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}]}, 'version': '0.6'}}


[<Image layer 'Overview_1: Brightfield' at 0x2814815d040>]

## Detect organoids

In [9]:
viewer.layers.clear()

folders = [
    r'Z:\Collaborations\aif-botton-collaboration\Johannes\20260911'
]

scenes = []
for folder in folders:
    scene_path = os.path.join(folder, "scene.ome.zarr")
    viewer.open(scene_path, plugin='napari-ome-zarr')
    scenes.append(OMEZarrScene.from_ome_zarr(scene_path))

for layer in viewer.layers:
    layer.auto_contrast = True

    layer.translate = layer.affine.translate
    layer.affine.translate = np.zeros_like(layer.translate)

Root group {'ome': {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': 'Overview_1'}, 'output': {'name': 'world'}, 'translation': [25.02000000000001, 2427.2011200000015, 2968.8107800000002]}], 'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}]}, 'version': '0.6'}}


In [10]:
resolution_level = 2
model = YOLO(r"C:\Users\aifadmin\Documents\Github\autoSCAPE\docs\notebooks\archive\runs\detect\train-6\weights\best.pt")
shapes = object_detection_on_scene(scenes=scenes, model=model, resolution_level=resolution_level)

shapes.face_color = [0,0,0,0]
shapes.edge_width = 10
shapes.units = viewer.layers[0].units
viewer.add_layer(shapes)

100%|██████████| 1/1 [00:23<00:00, 23.80s/it]


<Shapes layer 'shapes' at 0x2814864dac0>

c:\Users\aifadmin\Documents\Github\autoSCAPE\.venv\Lib\site-packages\biaplotter\colormap.py:34: UserWarning: Categorical colormap detected. Setting categorical=True. If the colormap is continuous, set categorical=False explicitly.
  warnings.warn(
c:\Users\aifadmin\Documents\Github\autoSCAPE\.venv\Lib\site-packages\biaplotter\artists.py:331: UserWarning: Color indices must be integers for categorical colormap. Change `overlay_colormap` to a continuous colormap or set `color_indices` to integers.
  warnings.warn(


## Focus detection

In [15]:
focus_model = FocusLightningModel.load_from_checkpoint(r"C:\Users\aifadmin\Documents\Github\autoSCAPE\docs\notebooks\archive\logs_focus_detector\lightning_logs\version_0\checkpoints\best.ckpt").eval()
focus_detections = run_focus_detection(viewer.layers["Overview_1: Brightfield"], detections=viewer.layers["shapes subcluster 1"], model=focus_model, crop_size=128)

Total parameters: 11,564,993
FC input dimension: 128


100%|██████████| 260/260 [02:01<00:00,  2.14it/s]


In [ ]:
viewer.add_layer(focus_detections)

## Convert SCAPE stacks

In [73]:
root = r"D:\UserData\Johannes\20260911\20260911_Spheroids_2x2_resScanner_longerExp"


deskewd_cs = CoordinateSystem(
    name="deskewed",
    axes=[
        Axis(name="c", type="channel"),
        Axis(name="z", type="space", unit="micrometer"),
        Axis(name="y", type="space", unit="micrometer"),
        Axis(name="x", type="space", unit="micrometer")
    ]
)

world_cs = CoordinateSystem(
    name="world",
    axes=[
        Axis(name="c", type="channel"),
        Axis(name="z", type="space", unit="micrometer"),
        Axis(name="y", type="space", unit="micrometer"),
        Axis(name="x", type="space", unit="micrometer")
    ]
)

tif_files = glob.glob(os.path.join(root, "**", "*.ome.tif"), recursive=True)
images = {("2", "C"): []}
translations = []

for idx, tf in tqdm.tqdm(enumerate(tif_files), total=len(tif_files), desc="Processing tiles"):
    metadata_file = tf.replace(".ome.tif", "_metadata.txt")
    with open(metadata_file) as f:
        metadata = yaml.safe_load(f)["Summary"]

    pos = Path(tf).stem.split(".")[0].split("_")[-1]

    translation = np.asarray(
        [
            0,
            float(metadata["Position_Y"].replace("µm", "").replace(",", "")),
            float(metadata["Position_X"].replace("µm", "").replace(",", ""))
        ]
    )

    translations.append(
        Translation(
            translation = translation.tolist(),
            input=CoordinateSystemIdentifier(name="deskewd", path=f"C/2/{idx}"),
            output=CoordinateSystemIdentifier(name="world")
        )
    )

    angle = float(metadata["StageScanAnglePathA"]) / 180 * np.pi
    # affine_matrix rows/cols follow the (c, z, y, x, 1) axis order used by
    # `img`/`oz_img` below (axes="czyx" + homogeneous row/col).
    # Row = output axis, column = input axis (matches napari's affine
    # convention: hom_coords @ affine_matrix.T).
    # Shearing y as a function of z (y' = y + shear * z) goes at
    # [row=y (index 2), col=z (index 1)].
    shear = 1 / np.tan(angle)
    affine_matrix = np.asarray(
        [
            [1, 0, 0, 0, 0],
            [0, 1, 0, 0, 0],
            [0, shear, 1, 0, 0],
            [0, 0, 0, 1, 0],
        ]
    )
    affine = Affine(
        affine=affine_matrix.tolist(),
        input=CoordinateSystemIdentifier(name="physical"),
        output=CoordinateSystemIdentifier(name="deskewed")
    )

    img = da.from_array(tff.imread(tf, is_ome=False, is_mmstack=False)).transpose((1, 0, 2, 3))

    oz_img = OMEZarrImage(
        img,
        axes="czyx",
        scale={"z": 2, "y": 0.391, "x": 0.391},
        axes_units={"z": "micrometer", "y": "micrometer", "x": "micrometer"},
        name=pos
        )

    oz_ms = OMEZarrMultiscale(
        oz_img,
        channel_names=["Far Red", "Actin"],
        channel_colors=["#00FFFF", "#FF00FF"],
        contrast_limits=[(37, 1500), (100, 400)],
        coordinate_systems=(deskewd_cs,),
        coordinate_transformations=(affine,),
        )

    images[("2", "C")].append(oz_ms)

Processing tiles:   0%|          | 0/260 [00:00<?, ?it/s]

<string>:9: UserWarning: Scale value not provided for ax(i)(e)s '{'c'}'. Using default scale of 1.0.
Processing tiles: 100%|██████████| 260/260 [1:14:52<00:00, 17.28s/it]


In [74]:
plate = OMEZarrHCSPlate(
    images=images
)

In [75]:
plate.to_ome_zarr(
    os.path.join(r"Z:\Collaborations\aif-botton-collaboration\Johannes\20260910", "organoids.ome.zarr"),
    compute=True, overwrite=True
    )

c:\Users\aifadmin\Documents\Github\autoSCAPE\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `tuple[Dataset, ...]` - serialized value may not be as expected [field_name='datasets', input_value=[Dataset(path='s0', coord....957725806451613)))),))], input_type=list])
  return self.__pydantic_serializer__.to_python(


[]